# Train Roman's LoRA on Kaggle (Z-Image Turbo, free)

Runs unattended with **Save Version → Save & Run All**: no clicking between cells.

- **Input:** the private Kaggle dataset made from `outputs/lora/dataset_<date>.zip`
  (`uv run ca lora prepare`): one folder of `NN.jpg` + `NN.txt` captions that start with the
  trigger word (e.g. `r0man_bg, a man`).
- **Settings:** Accelerator **GPU T4 x2**, Internet **on**.
- **Trainer:** Ostris AI Toolkit with its example defaults (rank 16, 2000 steps, lr 1e-4,
  adamw8bit) and its low-VRAM options (transformer and text encoder in float8, low_vram, cached
  latents and text embeddings). Z-Image Turbo is distilled, so it trains with Ostris' Turbo
  training adapter.
- **Output (`/kaggle/working`):** `lora_output/<job>/` with a checkpoint and sample images every
  250 steps, `train_log.txt`, `status.json`, and everything in one `<job>_outputs.zip`.
- **Time:** training stops after `TIME_BUDGET_HOURS` (well inside the session limit); every
  finished checkpoint is kept, so a stop loses at most the steps since the last one.

The photos stay in `/tmp` and are not part of the output.

In [ ]:
# Settings: AI Toolkit's example defaults; no tuning.
STEPS = 2000
RANK = 16  # linear and linear_alpha
LEARNING_RATE = 1e-4
SAVE_EVERY = 250  # a checkpoint + sample images every N steps
KEEP_CHECKPOINTS = 20  # keep all of them, to pick the best one later
TIME_BUDGET_HOURS = 9.0  # training stops here; the session limit is about 12 h
RESOLUTIONS = [512, 768, 1024]  # AI Toolkit default buckets
FALLBACK_RESOLUTIONS = [512, 768]  # used once if the GPU runs out of memory early
BASE_MODEL = "Tongyi-MAI/Z-Image-Turbo"
ADAPTER_REPO = "ostris/zimage_turbo_training_adapter"
TOOLKIT_REPO = "https://github.com/ostris/ai-toolkit.git"
WORK = "/kaggle/working"
OUTPUT_DIR = f"{WORK}/lora_output"
SCRATCH = "/tmp/ca_lora"  # dataset, toolkit and base models: not saved as output
SAMPLE_PROMPTS = [
    "[trigger], a man in a navy suit in a modern office, photorealistic portrait, window light",
    "[trigger], a man smiling at a cafe table with a laptop, natural light, photo",
    "[trigger], close-up portrait of a man, neutral grey background, studio light",
    "[trigger], a man presenting in front of a screen in a meeting room, photo",
    "a man in a navy suit in a modern office, photorealistic portrait, window light",
]

In [ ]:
import json
import os
import subprocess
import time
from pathlib import Path

import torch


def sh(command: str) -> None:
    """Run a shell command and stop the notebook if it fails (only used before training)."""
    print("$", command, flush=True)
    subprocess.run(command, shell=True, check=True)


if not torch.cuda.is_available():
    raise SystemExit("No GPU. Settings -> Accelerator -> GPU T4 x2, then Save & Run All again.")
MAJOR, MINOR = torch.cuda.get_device_capability(0)
# T4 (7.5) and P100 (6.0) have no native bf16: fp16 there, bf16 on newer GPUs.
DTYPE = "bf16" if MAJOR >= 8 else "fp16"
print(f"GPU: {torch.cuda.get_device_name(0)} (compute {MAJOR}.{MINOR}), "
      f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB, dtype {DTYPE}")


os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SCRATCH, exist_ok=True)
os.environ["HF_HOME"] = f"{SCRATCH}/hf"  # base models stay out of /kaggle/working
STATUS: dict[str, object] = {
    "status": "started", "dtype": DTYPE, "gpu": torch.cuda.get_device_name(0),
}


def save_status(**changes: object) -> None:
    STATUS.update(changes)
    Path(OUTPUT_DIR, "status.json").write_text(json.dumps(STATUS, indent=2))


save_status()

In [ ]:
# The dataset from `ca lora prepare`: a folder of image + caption pairs (Kaggle usually
# unzips an uploaded zip; a zip that is still packed is unpacked here).
import shutil
import zipfile

IMAGE_TYPES = {".jpg", ".jpeg", ".png", ".webp"}


def pairs(folder: Path) -> list[Path]:
    return sorted(
        image for image in folder.iterdir()
        if image.suffix.lower() in IMAGE_TYPES and image.with_suffix(".txt").is_file()
    )


for packed in Path("/kaggle/input").rglob("dataset_*.zip"):
    target = Path(SCRATCH, "unzipped", packed.stem)
    if not target.exists():
        with zipfile.ZipFile(packed) as bundle:
            bundle.extractall(target)
roots = [Path("/kaggle/input"), Path(SCRATCH, "unzipped")]
folders = [folder for root in roots if root.exists() for folder in [root, *root.rglob("*")]
           if folder.is_dir()]
found = max(folders, key=lambda folder: len(pairs(folder)), default=None)
if found is None or len(pairs(found)) < 9:
    save_status(status="failed: no dataset with at least 9 image + caption pairs")
    raise SystemExit("Attach the private dataset made from `ca lora prepare` (Add Input).")
DATASET = Path(SCRATCH, "dataset")
shutil.rmtree(DATASET, ignore_errors=True)
DATASET.mkdir(parents=True)
for image in pairs(found):
    shutil.copy(image, DATASET / image.name)
    shutil.copy(image.with_suffix(".txt"), DATASET / image.with_suffix(".txt").name)
captions = [path.read_text().strip() for path in sorted(DATASET.glob("*.txt"))]
TRIGGER = captions[0].split(",")[0].strip()
if not TRIGGER or any(not caption.startswith(TRIGGER) for caption in captions):
    save_status(status="failed: captions do not all start with the same trigger word")
    raise SystemExit("Every caption must start with the trigger word (make the dataset again).")
JOB = f"{TRIGGER}_z_image_turbo"
print(f"Dataset: {found} -> {len(captions)} images, trigger {TRIGGER!r}, job {JOB}")
save_status(dataset=str(found), images=len(captions), trigger=TRIGGER, job=JOB)

In [ ]:
# AI Toolkit (MIT): clone and install its requirements; Kaggle's preinstalled PyTorch is used.
import sys

TOOLKIT = Path(SCRATCH, "ai-toolkit")
if not (TOOLKIT / "run.py").exists():
    sh(f"git clone --depth 1 {TOOLKIT_REPO} {TOOLKIT}")
commit = subprocess.run(["git", "-C", str(TOOLKIT), "rev-parse", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
sh(f"{sys.executable} -m pip install -q -r {TOOLKIT}/requirements.txt")
save_status(toolkit_commit=commit)
print("AI Toolkit commit:", commit)

In [ ]:
# The Z-Image arch name as this toolkit version registers it, and the newest Turbo adapter.
import re

from huggingface_hub import list_repo_files

ARCH_PATTERN = re.compile(r"""arch\s*=\s*['"]([A-Za-z0-9_:.-]*z_?image[A-Za-z0-9_:.-]*)['"]""",
                          re.IGNORECASE)
arches = sorted({
    match.group(1)
    for path in TOOLKIT.rglob("*.py")
    for match in ARCH_PATTERN.finditer(path.read_text(errors="ignore"))
})
print("Z-Image arches in this AI Toolkit:", arches)
if not arches:
    save_status(status="failed: this AI Toolkit version has no Z-Image arch")
    raise SystemExit("No Z-Image support found in AI Toolkit (see status.json).")
ARCH = "zimage" if "zimage" in arches else arches[0]
adapters = sorted(name for name in list_repo_files(ADAPTER_REPO) if name.endswith(".safetensors"))
if not adapters:
    save_status(status=f"failed: no adapter file in {ADAPTER_REPO}")
    raise SystemExit(f"No training adapter found in {ADAPTER_REPO}.")
ADAPTER = f"{ADAPTER_REPO}/{adapters[-1]}"  # e.g. ..._v2 over ..._v1
print("Arch:", ARCH, "| adapter:", ADAPTER)
save_status(arch=ARCH, adapter=ADAPTER)

In [ ]:
# The AI Toolkit config (YAML; JSON strings and numbers are valid YAML scalars).
CONFIG = Path(OUTPUT_DIR, f"{JOB}.yaml")


def q(value: object) -> str:
    return json.dumps(value)


def write_config(resolutions: list[int]) -> None:
    prompts = "\n".join(f"          - {q(prompt)}" for prompt in SAMPLE_PROMPTS)
    CONFIG.write_text(f"""job: extension
config:
  name: {q(JOB)}
  process:
    - type: "sd_trainer"
      training_folder: {q(OUTPUT_DIR)}
      device: "cuda:0"
      trigger_word: {q(TRIGGER)}
      network:
        type: "lora"
        linear: {RANK}
        linear_alpha: {RANK}
      save:
        dtype: "float16"
        save_every: {SAVE_EVERY}
        max_step_saves_to_keep: {KEEP_CHECKPOINTS}
      datasets:
        - folder_path: {q(str(DATASET))}
          caption_ext: "txt"
          caption_dropout_rate: 0.05
          cache_latents_to_disk: true
          resolution: {q(resolutions)}
      train:
        batch_size: 1
        cache_text_embeddings: true
        steps: {STEPS}
        gradient_accumulation: 1
        train_unet: true
        train_text_encoder: false
        gradient_checkpointing: true
        noise_scheduler: "flowmatch"
        optimizer: "adamw8bit"
        lr: {LEARNING_RATE}
        dtype: {q(DTYPE)}
      model:
        name_or_path: {q(BASE_MODEL)}
        arch: {q(ARCH)}
        assistant_lora_path: {q(ADAPTER)}
        quantize: true
        qtype: "qfloat8"
        quantize_te: true
        qtype_te: "qfloat8"
        low_vram: true
      sample:
        sampler: "flowmatch"
        sample_every: {SAVE_EVERY}
        width: 1024
        height: 1024
        prompts:
{prompts}
        neg: ""
        seed: 42
        walk_seed: true
        guidance_scale: 1
        sample_steps: 8
meta:
  name: "[name]"
  version: "1.0"
""")


write_config(RESOLUTIONS)
print(CONFIG.read_text())

In [ ]:
# Train. Never raises: whatever happens, the next cell packs every finished checkpoint.
import signal

LOG = Path(OUTPUT_DIR, "train_log.txt")
DEADLINE = time.time() + TIME_BUDGET_HOURS * 3600


def checkpoints() -> list[Path]:
    return sorted(Path(OUTPUT_DIR, JOB).glob("*.safetensors"))


def run_training(resolutions: list[int]) -> str:
    write_config(resolutions)
    started = time.time()
    with LOG.open("a") as log:
        log.write(f"\n=== training at {resolutions} ===\n")
        log.flush()
        process = subprocess.Popen([sys.executable, "run.py", str(CONFIG)], cwd=TOOLKIT,
                                   stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
        shown = started
        while process.poll() is None:
            time.sleep(30)
            if time.time() > DEADLINE:
                os.killpg(process.pid, signal.SIGINT)  # AI Toolkit stops at Ctrl+C
                try:
                    process.wait(timeout=300)
                except subprocess.TimeoutExpired:
                    os.killpg(process.pid, signal.SIGKILL)
                return "stopped: time budget reached"
            if time.time() - shown > 600:  # progress every 10 minutes
                shown = time.time()
                tail = LOG.read_text(errors="ignore").splitlines()[-2:]
                print(f"[{(shown - started) / 3600:.1f} h] {len(checkpoints())} checkpoints;",
                      " | ".join(line[-160:] for line in tail), flush=True)
    if process.returncode == 0:
        return "done"
    text = LOG.read_text(errors="ignore")
    if "out of memory" in text.lower() and not checkpoints():
        return "oom"
    return f"failed: exit code {process.returncode} (see train_log.txt)"


try:
    result = run_training(RESOLUTIONS)
    if result == "oom":
        print("Out of GPU memory before the first checkpoint: trying once at", FALLBACK_RESOLUTIONS)
        STATUS["notes"] = ["out of memory at 1024: trained at 512/768"]
        result = run_training(FALLBACK_RESOLUTIONS)
        if result == "oom":
            result = "failed: out of GPU memory also at 512/768 (see train_log.txt)"
except Exception as error:  # noqa: BLE001 - the outputs are packed whatever happened
    result = f"failed: {type(error).__name__}: {error}"
save_status(status=result, checkpoints=[path.name for path in checkpoints()])
print("Training:", result)

In [ ]:
# Pack every checkpoint, sample image, the config, the log and status.json into one zip.
ARCHIVE = Path(WORK, f"{JOB}_outputs.zip")
with zipfile.ZipFile(ARCHIVE, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(Path(OUTPUT_DIR).rglob("*")):
        if path.is_file() and "_latent_cache" not in path.parts:
            bundle.write(path, path.relative_to(OUTPUT_DIR))
samples = sorted(Path(OUTPUT_DIR, JOB, "samples").glob("*")) if JOB else []
print(f"Status: {STATUS['status']}")
print(f"Checkpoints ({len(checkpoints())}):", *[path.name for path in checkpoints()], sep="\n  ")
print(f"Sample images: {len(samples)} in {Path(OUTPUT_DIR, JOB, 'samples')}")
print(f"Everything in one file: {ARCHIVE} ({ARCHIVE.stat().st_size / 2**20:.0f} MB)")
print("Next: pick a checkpoint from its samples, then run kaggle_test_lora.ipynb with it.")